# step-counter-increment — ex1: step counter increments AFTER optimizer.step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `step-counter-increment`. Running the final beacon cell reports progress against the `Trainer: step counter increment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: step counter increment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`step-counter-increment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "step-counter-increment"
DD_SUBTOPIC = "Trainer: step counter increment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `self.step += 1` placement — quick refresher

Every trainer / optimizer that needs a notion of TIME (logging intervals, learning-rate schedules, Adam's bias correction, wandb x-axis) carries a step counter. The canonical placement:

```
loss = self._step(x, y)
loss.backward()
self.optimizer.step()        # apply update
self.optimizer.zero_grad()   # clear grads
self.step += 1               # tick AFTER the update is committed
if self.step % LOG_EVERY == 0:
    self.log({'loss': loss.item(), 'step': self.step})
```

**Why AFTER optimizer.step, not before.** The step counter measures how many UPDATES have been applied to the model. Incrementing before the update would mean step 1 logs the model state from BEFORE step 1 ran. Off-by-one bugs in training graphs almost always trace back to this placement.

**Why BEFORE logging.** Logging at step N should reflect the state AFTER N updates have happened. So: update → increment → log.

**Adam's `self.t` is a SEPARATE counter** — internal to the optimizer, used for bias correction. The trainer's `self.step` is external. They happen to be equal if there's one trainer and one optimizer, but conceptually they're independent.

### Exercise 1 — step counter increments AFTER optimizer.step

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the canonical `self.step += 1` placement inside a training loop body: AFTER `optimizer.step()` and `zero_grad()`, BEFORE any logging that should reflect the post-update state.
> Keywords: step-counter, training-loop, logging, order-of-ops
> ```

**KCs targeted:** `step-counter-increments-after-optimizer-step`, `step-counter-starts-at-zero`

Implement `ex1_train_one_epoch(model, optimizer, loader, loss_fn, start_step)`. ONE epoch over the loader with a correctly-placed step counter.

Initialize `step = start_step`. For each `(x, y)` in `loader`:

1. `loss = loss_fn(model(x), y)` — forward + loss.
2. `loss.backward()` — compute grads.
3. `optimizer.step()` — apply update.
4. `optimizer.zero_grad()` — clear grads.
5. `step += 1` — tick AFTER the update is committed.
6. Record `(step, loss.item())` in a log list AFTER the tick — so the log entry's step value reflects the state AFTER `step` batches have been processed.

Return `(final_step, log_list)`.

Inputs:
- `model`, `optimizer`, `loader`, `loss_fn`: usual.
- `start_step`: int — the counter value BEFORE this epoch runs. (Lets you accumulate across epochs.)

Output:
- `final_step`: int — the counter after the epoch.
- `log_list`: list of `(step, loss_float)` tuples, one per batch.

In [ ]:
def ex1_train_one_epoch(model, optimizer, loader, loss_fn, start_step):
    step = start_step
    log = []
    for x, y in loader:
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        step += 1
        log.append((step, loss.item()))
    return step, log


<details><summary>Solution</summary>

```python
def ex1_train_one_epoch(model, optimizer, loader, loss_fn, start_step):
    step = start_step
    log = []
    for x, y in loader:
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        step += 1
        log.append((step, loss.item()))
    return step, log
```

**Why this placement matters.** Logging at step N should reflect 'N updates have been applied to the model.' If you log BEFORE incrementing, the first entry says step=0 — but an update DID happen before that line. The off-by-one ripples into every downstream graph: learning-rate schedules trigger one step late, checkpoints save the wrong state, wandb curves are misaligned across runs.

**Why start_step is a parameter.** A training run typically calls `train_one_epoch` in a loop. Each epoch must continue the global counter, not reset to zero — otherwise you'd overwrite step-0 logs every epoch. Passing `start_step` explicitly makes the contract clear.

**ARENA chap-3 specifically.** The transformer-training code in chap-3 follows this exact placement: `step += 1` after `optimizer.step() / optimizer.zero_grad()`, before any `wandb.log(...)` call. The Trainer pattern from trainer-class-skeleton (sibling drill) puts the same line inside `fit()`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()